# 03 Higher-Order Effects
Ablation-focused checks for temporal clusters.


## Setup


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from notebook_runner import run_notebook_script


## Configuration


In [ ]:
SCRIPT_NAME = '03_higher_order_effects'

overrides = {
    'dataset_name': 'temporal_clusters',
    'dataset_kwargs': {},
    'render_all_plots': False,  # generate each figure below
    # 'run_name': 'temporal_clusters_dbgnn',
}


## Compute Shared State


In [ ]:
state = run_notebook_script(SCRIPT_NAME, overrides=overrides)
print('Plot dir:', state['PLOT_DIR'])
print('Rows in HO ablation table:', len(state['posthoc_df']))

palette = list(state['DEFAULT_CLASS_COLORS'])
print('Palette:', {'eventblue': palette[0], 'snapshotorange': palette[1], 'edgegray': state['EDGE_GRAY']})

### Plot 1: Lookup Table in HO Hidden Space (PCA)
HO branch self-loop-only pass, colored by class.


In [ ]:
Z = np.asarray(state['Z'])
y_ho = np.asarray(state['y_ho'])
classes = np.asarray(state['classes'])
palette = list(state['DEFAULT_CLASS_COLORS'])
plot_dir = Path(state['PLOT_DIR'])
dataset_name = str(state['dataset_name'])

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for i, cls in enumerate(classes):
    mask = y_ho == cls
    ax.scatter(
        Z[mask, 0],
        Z[mask, 1],
        s=22,
        alpha=0.82,
        color=palette[i % len(palette)],
        label=f'class {int(cls)}',
    )

ax.set_title('Lookup table in hidden space: HO self-loop-only pass (PCA)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(alpha=0.25, linestyle='--', linewidth=0.6, color=state['EDGE_GRAY'])
ax.legend(title='Class', frameon=False, ncol=min(4, len(classes)))
plt.tight_layout()
fig.savefig(plot_dir / f'lookup_table_pca_{dataset_name}.pdf', bbox_inches='tight')
plt.show()


### Plot 2: HO Ablation Summary Table
Print, display, and export table (CSV + PDF).


In [ ]:
posthoc_df = state['posthoc_df'].copy()
plot_dir = Path(state['PLOT_DIR'])
dataset_name = str(state['dataset_name'])

print('\nHO ablation summary:')
print(posthoc_df.to_string(index=False))
try:
    from IPython.display import display
    display(posthoc_df)
except Exception:
    pass

posthoc_df.to_csv(plot_dir / f'ho_ablation_summary_{dataset_name}.csv', index=False)

fig_tbl, ax_tbl = plt.subplots(figsize=(10, 0.6 + 0.42 * len(posthoc_df)))
ax_tbl.axis('off')
tbl = ax_tbl.table(
    cellText=posthoc_df.values,
    colLabels=posthoc_df.columns,
    cellLoc='center',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 1.25)
plt.tight_layout()
fig_tbl.savefig(plot_dir / f'ho_ablation_summary_{dataset_name}.pdf', bbox_inches='tight')
plt.show()
